# Stochastic Processes, White Noise, PSD, and Langevin Dynamics
## A computational tutorial for the stochastic-systems sections of *KalmanFilters_1.pdf*

This notebook develops five ideas that lead directly into Kalman filtering:

1. vector random (stochastic) processes;
2. white noise;
3. power spectral density;
4. discrete-time dynamic systems with random inputs;
5. stochastic differential equations and the Langevin equation, including the probability density function evolving over time.

The code uses Monte Carlo ensembles to connect individual random trajectories with means, covariance matrices, autocorrelation functions, spectra, and probability densities.


## Concept map

| Object | Domain | What it describes |
|---|---|---|
| Realization $x_k(\omega)$ | Time | One possible trajectory |
| Mean $m_k=\mathbb E[X_k]$ | Time | Ensemble center at each time |
| Covariance $C_{k,\ell}$ | Two time indices | Dependence between different times/components |
| Autocovariance $R_X[r]$ | Time lag | Dependence as a function of separation |
| PSD $S_X(f)$ | Frequency | Distribution of average power over frequency |
| State covariance $P_k$ | Time | Uncertainty propagated by a dynamic model |
| PDF $p(x,t)$ | State and time | Distribution of an ensemble as it evolves |

> A trajectory, a PDF, and a PSD are different views of the same stochastic phenomenon. A single realization does not by itself reveal the full probability law.


## Setup

Only NumPy and Plotly are required:

```bash
pip install numpy plotly
```


In [2]:
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots

np.set_printoptions(precision=4, suppress=True)
rng = np.random.default_rng(seed=2026)
integrate = np.trapezoid if hasattr(np, "trapezoid") else np.trapz


def normal_pdf(x, mean, std):
    """Evaluate a scalar Gaussian PDF; std may be broadcastable."""
    x = np.asarray(x, dtype=float)
    std = np.asarray(std, dtype=float)
    if np.any(std <= 0):
        raise ValueError('Every standard deviation must be positive.')
    return np.exp(-0.5 * ((x - mean) / std) ** 2) / (np.sqrt(2 * np.pi) * std)


# 1. Vector random (stochastic) processes

A vector stochastic process is a family of random vectors indexed by time:

$$\{X_k:k=0,1,2,\ldots\},\qquad X_k\in\mathbb R^n.
$$

There are two complementary viewpoints:

- Fix an outcome $\omega$: $x_0(\omega),x_1(\omega),\ldots$ is one **sample path** or realization.
- Fix a time $k$: $X_k$ is a random vector described by a distribution over an **ensemble** of possible systems.

The first two moments are

$$m_k=\mathbb E[X_k],$$

and

$$C_{k,\ell}=\mathbb E[(X_k-m_k)(X_\ell-m_\ell)^\top].$$

A process is wide-sense stationary when its mean is constant and its covariance depends only on the lag $r=k-\ell$. Strict stationarity is stronger: all finite-dimensional distributions must be invariant to a common time shift.

## Example: a two-dimensional stable stochastic recursion

We simulate

$$X_{k+1}=FX_k+W_k,\qquad W_k\sim\mathcal N(0,Q).$$

The deterministic initial condition makes the process initially nonstationary. Because $F$ is stable, its mean decays and its covariance approaches a steady value.


In [3]:
F_vec = np.array([[0.92, 0.18],
                  [-0.08, 0.85]])
Q_vec = np.array([[0.08, 0.02],
                  [0.02, 0.05]])
x0_vec = np.array([2.0, -1.0])
n_steps_vec = 160
n_ensemble_vec = 2_000
L_Q_vec = np.linalg.cholesky(Q_vec)

X_vec = np.empty((n_ensemble_vec, 2, n_steps_vec))
X_vec[:, :, 0] = x0_vec
for k in range(n_steps_vec - 1):
    noise_k = rng.standard_normal((n_ensemble_vec, 2)) @ L_Q_vec.T
    X_vec[:, :, k + 1] = X_vec[:, :, k] @ F_vec.T + noise_k

mean_vec = X_vec.mean(axis=0)
var_vec = X_vec.var(axis=0, ddof=1)
time_vec = np.arange(n_steps_vec)

# Fixed-point iteration for the stationary covariance P = F P F^T + Q.
P_stationary = np.zeros((2, 2))
for _ in range(10_000):
    P_next = F_vec @ P_stationary @ F_vec.T + Q_vec
    if np.allclose(P_next, P_stationary, atol=1e-13, rtol=0.0):
        break
    P_stationary = P_next

print('Eigenvalues of F:', np.linalg.eigvals(F_vec))
print('Final empirical mean:', mean_vec[:, -1])
print('Stationary covariance:\n', P_stationary)
print('Final empirical covariance:\n', np.cov(X_vec[:, :, -1], rowvar=False))

assert np.all(np.abs(np.linalg.eigvals(F_vec)) < 1.0)
assert np.allclose(np.cov(X_vec[:, :, -1], rowvar=False), P_stationary, atol=0.08)

fig_process = make_subplots(
    rows=2, cols=2,
    subplot_titles=('Sample paths of X₁', 'Sample paths of X₂',
                    'Ensemble means', 'Ensemble variances')
)
for j in range(18):
    fig_process.add_trace(
        go.Scatter(x=time_vec, y=X_vec[j, 0], mode='lines',
                   line=dict(width=1), opacity=0.35, showlegend=False),
        row=1, col=1
    )
    fig_process.add_trace(
        go.Scatter(x=time_vec, y=X_vec[j, 1], mode='lines',
                   line=dict(width=1), opacity=0.35, showlegend=False),
        row=1, col=2
    )

fig_process.add_trace(go.Scatter(x=time_vec, y=mean_vec[0], mode='lines',
                                 line=dict(width=3), name='E[X₁]'), row=2, col=1)
fig_process.add_trace(go.Scatter(x=time_vec, y=mean_vec[1], mode='lines',
                                 line=dict(width=3), name='E[X₂]'), row=2, col=1)
fig_process.add_trace(go.Scatter(x=time_vec, y=var_vec[0], mode='lines',
                                 line=dict(width=3), name='Var(X₁)'), row=2, col=2)
fig_process.add_trace(go.Scatter(x=time_vec, y=var_vec[1], mode='lines',
                                 line=dict(width=3), name='Var(X₂)'), row=2, col=2)
fig_process.add_hline(y=P_stationary[0, 0], line_dash='dash', line_color='royalblue',
                      row=2, col=2)
fig_process.add_hline(y=P_stationary[1, 1], line_dash='dash', line_color='darkorange',
                      row=2, col=2)
fig_process.update_xaxes(title_text='Time step k')
fig_process.update_layout(
    title='A vector stochastic process: trajectories and ensemble statistics',
    template='plotly_white', width=1_050, height=760
)
fig_process.show()


Eigenvalues of F: [0.885+0.1148j 0.885-0.1148j]
Final empirical mean: [-0.0237 -0.0014]
Stationary covariance:
 [[0.5991 0.0182]
 [0.0182 0.1851]]
Final empirical covariance:
 [[0.6214 0.0216]
 [0.0216 0.1812]]


### Reading the result

Each thin curve is one valid realization. The ensemble mean is calculated vertically across many realizations at a fixed time, not by averaging one trajectory over time. Those two averages agree only under additional conditions such as ergodicity.

The stable dynamics forget the deterministic initial condition. The mean approaches zero and the covariance approaches the solution of the discrete Lyapunov equation

$$P_\infty=F P_\infty F^\top+Q.
$$


# 2. White noise

A zero-mean discrete-time vector white-noise process satisfies

$$\mathbb E[W_k]=0,$$

and

$$\mathbb E[W_kW_\ell^\top]=Q\,\delta_{k\ell},$$

where $\delta_{k\ell}=1$ if $k=\ell$ and 0 otherwise. Thus, different time samples are uncorrelated.

Important distinctions:

- *White* describes second-order temporal structure; it does not require a Gaussian amplitude distribution.
- *Gaussian white noise* has jointly Gaussian samples. In the Gaussian case, zero cross-correlation implies independence.
- Components at the same time may still be correlated when $Q$ is not diagonal.
- Ideal white noise has infinite bandwidth and is an abstraction; sampled physical sensors are always bandwidth-limited.

We estimate the scalar autocovariance

$$\hat R[r]=\frac{1}{N}\sum_{k=0}^{N-r-1}(w_k-\bar w)(w_{k+r}-\bar w).$$


In [4]:
def biased_autocovariance(signal, max_lag):
    signal = np.asarray(signal, dtype=float)
    centered = signal - signal.mean()
    N = centered.size
    return np.array([
        centered[:N - lag] @ centered[lag:] / N
        for lag in range(max_lag + 1)
    ])

N_white = 8_192
Q_white = 1.5
w = np.sqrt(Q_white) * rng.standard_normal(N_white)
max_lag = 50
R_hat = biased_autocovariance(w, max_lag)
rho_hat = R_hat / R_hat[0]
lags = np.arange(max_lag + 1)
approx_bound = 1.96 / np.sqrt(N_white)

print(f'Sample mean: {w.mean():.5f}')
print(f'Sample variance: {w.var(ddof=1):.5f}; target Q={Q_white:.5f}')
print(f'Largest |sample ACF| outside lag 0: {np.max(np.abs(rho_hat[1:])):.4f}')
print(f'Approximate 95% reference bound: ±{approx_bound:.4f}')

fig_white = make_subplots(
    rows=1, cols=2,
    subplot_titles=('One realization', 'Estimated autocorrelation')
)
fig_white.add_trace(
    go.Scatter(x=np.arange(350), y=w[:350], mode='lines', name='w[k]'),
    row=1, col=1
)
fig_white.add_trace(
    go.Bar(x=lags, y=rho_hat, name='Sample ACF'),
    row=1, col=2
)
fig_white.add_hline(y=approx_bound, line_dash='dash', line_color='red', row=1, col=2)
fig_white.add_hline(y=-approx_bound, line_dash='dash', line_color='red', row=1, col=2)
fig_white.update_xaxes(title_text='Time step k', row=1, col=1)
fig_white.update_xaxes(title_text='Lag r', row=1, col=2)
fig_white.update_yaxes(title_text='Amplitude', row=1, col=1)
fig_white.update_yaxes(title_text='Correlation', row=1, col=2)
fig_white.update_layout(
    title='Gaussian white noise in the time domain',
    template='plotly_white', width=1_020, height=470, showlegend=False
)
fig_white.show()


Sample mean: 0.00207
Sample variance: 1.51359; target Q=1.50000
Largest |sample ACF| outside lag 0: 0.0202
Approximate 95% reference bound: ±0.0217


# 3. Power spectral density

For a zero-mean wide-sense stationary discrete process, the Wiener–Khinchin relation connects autocovariance and power spectral density:

$$S_X(f)=\sum_{r=-\infty}^{\infty}R_X[r]e^{-j2\pi fr},$$

with inverse

$$R_X[r]=\int_{-1/2}^{1/2}S_X(f)e^{j2\pi fr}\,df.
$$

Here $f$ is measured in cycles/sample. In particular,

$$R_X[0]=\operatorname{Var}(X_k)=\int_{-1/2}^{1/2}S_X(f)\,df.
$$

For white noise, $R_W[r]=Q\delta[r]$, so $S_W(f)=Q$ is flat. Passing white noise through dynamics creates colored noise. For the scalar AR(1) system

$$Y_k=aY_{k-1}+W_k,$$

the PSD is

$$S_Y(f)=\frac{Q}{|1-ae^{-j2\pi f}|^2}.
$$

We average periodograms across an ensemble to reduce the high variance of a single periodogram.


In [5]:
def ensemble_periodogram(signals):
    """Two-sided ensemble-averaged periodogram in cycles/sample."""
    signals = np.asarray(signals, dtype=float)
    N = signals.shape[-1]
    centered = signals - signals.mean(axis=-1, keepdims=True)
    spectrum = np.fft.fft(centered, axis=-1)
    psd = np.mean(np.abs(spectrum) ** 2 / N, axis=0)
    frequency = np.fft.fftfreq(N, d=1.0)
    return np.fft.fftshift(frequency), np.fft.fftshift(psd)

n_records = 300
N_psd = 1_024
Q_psd = 1.0
a_ar = 0.92

white_records = np.sqrt(Q_psd) * rng.standard_normal((n_records, N_psd))
colored_records = np.empty_like(white_records)
colored_records[:, 0] = rng.normal(
    0.0, np.sqrt(Q_psd / (1.0 - a_ar ** 2)), size=n_records
)
for k in range(1, N_psd):
    colored_records[:, k] = a_ar * colored_records[:, k - 1] + white_records[:, k]

f_psd, S_white_hat = ensemble_periodogram(white_records)
_, S_colored_hat = ensemble_periodogram(colored_records)
S_white_theory = np.full_like(f_psd, Q_psd)
S_colored_theory = Q_psd / np.abs(1.0 - a_ar * np.exp(-1j * 2.0 * np.pi * f_psd)) ** 2
df = 1.0 / N_psd

print(f'White-noise power from PSD: {np.sum(S_white_hat) * df:.4f}')
print(f'White-noise target variance: {Q_psd:.4f}')
print(f'AR(1) power from PSD:        {np.sum(S_colored_hat) * df:.4f}')
print(f'AR(1) theoretical variance:  {Q_psd / (1-a_ar**2):.4f}')

assert np.isclose(np.sum(S_white_hat) * df, Q_psd, rtol=0.03)
assert np.isclose(np.sum(S_colored_hat) * df, Q_psd / (1-a_ar**2), rtol=0.08)

fig_psd = make_subplots(
    rows=1, cols=2,
    subplot_titles=('White noise: flat spectrum', 'AR(1): low-pass colored spectrum')
)
fig_psd.add_trace(go.Scatter(x=f_psd, y=S_white_hat, mode='lines',
                             name='Estimated white PSD'), row=1, col=1)
fig_psd.add_trace(go.Scatter(x=f_psd, y=S_white_theory, mode='lines',
                             line=dict(color='black', dash='dash', width=3),
                             name='White theory'), row=1, col=1)
fig_psd.add_trace(go.Scatter(x=f_psd, y=S_colored_hat, mode='lines',
                             name='Estimated AR(1) PSD'), row=1, col=2)
fig_psd.add_trace(go.Scatter(x=f_psd, y=S_colored_theory.real, mode='lines',
                             line=dict(color='black', dash='dash', width=3),
                             name='AR(1) theory'), row=1, col=2)
fig_psd.update_xaxes(title_text='Frequency [cycles/sample]')
fig_psd.update_yaxes(title_text='PSD')
fig_psd.update_layout(
    title='Power spectral density: white versus dynamically colored noise',
    template='plotly_white', width=1_050, height=500
)
fig_psd.show()


White-noise power from PSD: 1.0003
White-noise target variance: 1.0000
AR(1) power from PSD:        6.2627
AR(1) theoretical variance:  6.5104


# 4. Discrete-time dynamic systems with random inputs

Consider the linear stochastic state-space model

$$X_{k+1}=F_kX_k+B_ku_k+G_kW_k,$$

$$Z_k=H_kX_k+V_k,$$

where

$$W_k\sim\mathcal N(0,Q_k),\qquad V_k\sim\mathcal N(0,R_k).$$

Assuming $W_k$ is independent of $X_k$, the state mean and covariance propagate as

$$m_{k+1}=F_km_k+B_ku_k,$$

$$P_{k+1}=F_kP_kF_k^\top+G_kQ_kG_k^\top.
$$

These are the Kalman filter **prediction equations** before a measurement update. The key difference from deterministic simulation is that we propagate a distribution, summarized here by its first two moments.

## Example: constant-velocity motion driven by random acceleration

With $X_k=[p_k,v_k]^\top$ and sampling interval $\Delta t$,

$$
F=\begin{bmatrix}1&\Delta t\\0&1\end{bmatrix},\qquad
G=\begin{bmatrix}\Delta t^2/2\\\Delta t\end{bmatrix}.
$$

A scalar random acceleration $W_k$ enters position and velocity through $G$. Position measurements are corrupted by $V_k$.


In [6]:
dt_dyn = 0.1
F_dyn = np.array([[1.0, dt_dyn],
                  [0.0, 1.0]])
G_dyn = np.array([[0.5 * dt_dyn ** 2],
                  [dt_dyn]])
H_dyn = np.array([[1.0, 0.0]])
q_acceleration = 0.8
R_measurement = 4.0
m0_dyn = np.array([0.0, 5.0])
P0_dyn = np.diag([1.0, 0.25])
n_steps_dyn = 151
n_ensemble_dyn = 5_000
time_dyn = np.arange(n_steps_dyn) * dt_dyn

# Monte Carlo ensemble.
states = m0_dyn + rng.standard_normal((n_ensemble_dyn, 2)) @ np.linalg.cholesky(P0_dyn).T
state_history_display = np.empty((25, 2, n_steps_dyn))
state_history_display[:, :, 0] = states[:25]
empirical_mean = np.empty((2, n_steps_dyn))
empirical_cov = np.empty((2, 2, n_steps_dyn))
empirical_mean[:, 0] = states.mean(axis=0)
empirical_cov[:, :, 0] = np.cov(states, rowvar=False)

for k in range(n_steps_dyn - 1):
    random_acceleration = np.sqrt(q_acceleration) * rng.standard_normal(n_ensemble_dyn)
    states = states @ F_dyn.T + random_acceleration[:, None] * G_dyn.ravel()
    state_history_display[:, :, k + 1] = states[:25]
    empirical_mean[:, k + 1] = states.mean(axis=0)
    empirical_cov[:, :, k + 1] = np.cov(states, rowvar=False)

# Analytical moment propagation.
mean_pred = np.empty((2, n_steps_dyn))
P_pred = np.empty((2, 2, n_steps_dyn))
mean_pred[:, 0] = m0_dyn
P_pred[:, :, 0] = P0_dyn
Q_state = (G_dyn * q_acceleration) @ G_dyn.T
for k in range(n_steps_dyn - 1):
    mean_pred[:, k + 1] = F_dyn @ mean_pred[:, k]
    P_pred[:, :, k + 1] = F_dyn @ P_pred[:, :, k] @ F_dyn.T + Q_state

# One noisy measurement sequence, shown only for interpretation.
true_position = state_history_display[0, 0]
measurements = true_position + np.sqrt(R_measurement) * rng.standard_normal(n_steps_dyn)
sigma_position = np.sqrt(P_pred[0, 0])
upper_position = mean_pred[0] + 2.0 * sigma_position
lower_position = mean_pred[0] - 2.0 * sigma_position

final_position_var_error = abs(empirical_cov[0, 0, -1] - P_pred[0, 0, -1]) / P_pred[0, 0, -1]
final_velocity_var_error = abs(empirical_cov[1, 1, -1] - P_pred[1, 1, -1]) / P_pred[1, 1, -1]
print('Final predicted mean:', mean_pred[:, -1])
print('Final empirical mean:', empirical_mean[:, -1])
print('Final predicted covariance:\n', P_pred[:, :, -1])
print('Final empirical covariance:\n', empirical_cov[:, :, -1])
print(f'Relative position-variance error: {final_position_var_error:.2%}')
print(f'Relative velocity-variance error: {final_velocity_var_error:.2%}')

assert final_position_var_error < 0.06
assert final_velocity_var_error < 0.06

fig_dyn = make_subplots(
    rows=2, cols=2, specs=[[{'colspan': 2}, None], [{}, {}]],
    subplot_titles=('One realization, measurements, and predicted uncertainty',
                    'Position variance', 'Velocity variance')
)
fig_dyn.add_trace(go.Scatter(x=time_dyn, y=upper_position, mode='lines',
                             line=dict(width=0), showlegend=False), row=1, col=1)
fig_dyn.add_trace(go.Scatter(x=time_dyn, y=lower_position, mode='lines',
                             fill='tonexty', fillcolor='rgba(65,105,225,0.18)',
                             line=dict(width=0), name='Predicted ±2σ'), row=1, col=1)
fig_dyn.add_trace(go.Scatter(x=time_dyn, y=mean_pred[0], mode='lines',
                             line=dict(color='royalblue', width=3),
                             name='Predicted mean'), row=1, col=1)
fig_dyn.add_trace(go.Scatter(x=time_dyn, y=true_position, mode='lines',
                             line=dict(color='black', width=2),
                             name='One true trajectory'), row=1, col=1)
fig_dyn.add_trace(go.Scatter(x=time_dyn, y=measurements, mode='markers',
                             marker=dict(size=4, opacity=0.45, color='crimson'),
                             name='Noisy measurements'), row=1, col=1)
fig_dyn.add_trace(go.Scatter(x=time_dyn, y=P_pred[0, 0], mode='lines',
                             line=dict(width=3), name='Predicted P₁₁'), row=2, col=1)
fig_dyn.add_trace(go.Scatter(x=time_dyn, y=empirical_cov[0, 0], mode='lines',
                             line=dict(width=2, dash='dash'), name='Empirical Var(p)'), row=2, col=1)
fig_dyn.add_trace(go.Scatter(x=time_dyn, y=P_pred[1, 1], mode='lines',
                             line=dict(width=3), name='Predicted P₂₂'), row=2, col=2)
fig_dyn.add_trace(go.Scatter(x=time_dyn, y=empirical_cov[1, 1], mode='lines',
                             line=dict(width=2, dash='dash'), name='Empirical Var(v)'), row=2, col=2)
fig_dyn.update_xaxes(title_text='Time [s]')
fig_dyn.update_yaxes(title_text='Position', row=1, col=1)
fig_dyn.update_yaxes(title_text='Variance', row=2, col=1)
fig_dyn.update_yaxes(title_text='Variance', row=2, col=2)
fig_dyn.update_layout(
    title='Random inputs propagate into state uncertainty',
    template='plotly_white', width=1_050, height=780
)
fig_dyn.show()


Final predicted mean: [75.  5.]
Final empirical mean: [74.8311  4.9868]
Final predicted covariance:
 [[147.249  12.75 ]
 [ 12.75    1.45 ]]
Final empirical covariance:
 [[152.327   13.3334]
 [ 13.3334   1.5173]]
Relative position-variance error: 3.45%
Relative velocity-variance error: 4.64%


### Interpretation

The mean follows the noise-free constant-velocity trajectory because the process noise has zero mean. The covariance grows because every acceleration disturbance adds uncertainty and the dynamics carry earlier velocity uncertainty into position.

The red measurements are displayed but are **not used** in this simulation to correct the state. Therefore, this is stochastic prediction, not yet a complete Kalman filter. A measurement update would condition the state distribution on each observed measurement and typically reduce $P_k$.


# 5. Stochastic systems and the Langevin equation

A continuous-time Langevin model is often written formally as

$$\frac{dX}{dt}=a(X,t)+b(X,t)\,\xi(t),$$

where $\xi(t)$ is ideal white noise. More rigorously, this is the stochastic differential equation (SDE)

$$dX_t=a(X_t,t)dt+b(X_t,t)dW_t,$$

where $W_t$ is Brownian motion. White noise is treated as the generalized derivative of Brownian motion; it is not simulated as a finite-valued pointwise function. Over a small interval,

$$\Delta W_k\sim\mathcal N(0,\Delta t).$$

The Euler–Maruyama method is

$$X_{k+1}=X_k+a(X_k,t_k)\Delta t+b(X_k,t_k)\sqrt{\Delta t}Z_k,
\qquad Z_k\sim\mathcal N(0,1).$$

## Ornstein–Uhlenbeck Langevin equation

We use the mean-reverting model

$$dX_t=-\lambda(X_t-\theta)dt+\sigma dW_t.
$$

The drift pulls the state toward $\theta$ while diffusion continually spreads the ensemble. Starting from deterministic $X_0=x_0$, the exact distribution is Gaussian:

$$m(t)=\theta+(x_0-\theta)e^{-\lambda t},$$

$$v(t)=\frac{\sigma^2}{2\lambda}\left(1-e^{-2\lambda t}\right).$$


In [7]:
lambda_ou = 1.2
theta_ou = 0.0
sigma_ou = 0.8
x0_ou = 3.0
dt_ou = 0.01
T_ou = 5.0
n_steps_ou = int(T_ou / dt_ou)
n_paths_ou = 30_000
time_ou = np.arange(n_steps_ou + 1) * dt_ou

x_current = np.full(n_paths_ou, x0_ou)
n_display = 30
paths_ou = np.empty((n_display, n_steps_ou + 1))
paths_ou[:, 0] = x0_ou
snapshot_times = [0.25, 1.0, 3.0, 5.0]
snapshot_indices = {int(round(t / dt_ou)): t for t in snapshot_times}
snapshots_ou = {}

for k in range(n_steps_ou):
    dW = np.sqrt(dt_ou) * rng.standard_normal(n_paths_ou)
    x_current += -lambda_ou * (x_current - theta_ou) * dt_ou + sigma_ou * dW
    paths_ou[:, k + 1] = x_current[:n_display]
    if (k + 1) in snapshot_indices:
        snapshots_ou[snapshot_indices[k + 1]] = x_current.copy()

mean_ou = theta_ou + (x0_ou - theta_ou) * np.exp(-lambda_ou * time_ou)
var_ou = sigma_ou ** 2 / (2.0 * lambda_ou) * (1.0 - np.exp(-2.0 * lambda_ou * time_ou))
std_ou = np.sqrt(var_ou)

final_mean_error = abs(x_current.mean() - mean_ou[-1])
final_var_error = abs(x_current.var(ddof=1) - var_ou[-1])
print(f'Final theoretical mean: {mean_ou[-1]:.5f}')
print(f'Final Monte Carlo mean: {x_current.mean():.5f}')
print(f'Final theoretical variance: {var_ou[-1]:.5f}')
print(f'Final Monte Carlo variance: {x_current.var(ddof=1):.5f}')
print(f'Stationary variance σ²/(2λ): {sigma_ou**2/(2*lambda_ou):.5f}')

assert final_mean_error < 0.02
assert final_var_error < 0.02

fig_ou_paths = go.Figure()
for j in range(n_display):
    fig_ou_paths.add_trace(go.Scatter(
        x=time_ou, y=paths_ou[j], mode='lines',
        line=dict(width=1), opacity=0.3, showlegend=False
    ))
fig_ou_paths.add_trace(go.Scatter(
    x=time_ou, y=mean_ou + 2.0 * std_ou, mode='lines',
    line=dict(width=0), showlegend=False
))
fig_ou_paths.add_trace(go.Scatter(
    x=time_ou, y=mean_ou - 2.0 * std_ou, mode='lines',
    fill='tonexty', fillcolor='rgba(65,105,225,0.18)',
    line=dict(width=0), name='Analytical ±2σ'
))
fig_ou_paths.add_trace(go.Scatter(
    x=time_ou, y=mean_ou, mode='lines',
    line=dict(color='black', width=4), name='Analytical mean'
))
fig_ou_paths.update_layout(
    title='Ornstein–Uhlenbeck sample paths and analytical moments',
    xaxis_title='Time t', yaxis_title='State X(t)',
    template='plotly_white', width=950, height=560
)
fig_ou_paths.show()


Final theoretical mean: 0.00744
Final Monte Carlo mean: 0.01248
Final theoretical variance: 0.26667
Final Monte Carlo variance: 0.27233
Stationary variance σ²/(2λ): 0.26667


## From the Langevin equation to a PDF: the Fokker–Planck equation

The SDE describes random paths. The corresponding probability density satisfies the Fokker–Planck equation

$$
\frac{\partial p}{\partial t}
=-\frac{\partial}{\partial x}[a(x,t)p(x,t)]
+\frac{1}{2}\frac{\partial^2}{\partial x^2}[b^2(x,t)p(x,t)].
$$

For the Ornstein–Uhlenbeck process,

$$
\frac{\partial p}{\partial t}
=\lambda\frac{\partial}{\partial x}[(x-\theta)p]
+\frac{\sigma^2}{2}\frac{\partial^2p}{\partial x^2}.
$$

The next surface is the analytical PDF $p(x,t)$. It begins concentrated near $x_0$, its center moves toward $\theta$, and its width approaches the stationary standard deviation $\sigma/\sqrt{2\lambda}$. We start the plot at $t=\Delta t$ because a deterministic initial condition is a Dirac delta at $t=0$, not an ordinary finite-height density.


In [8]:
t_pdf = np.linspace(dt_ou, T_ou, 220)
x_pdf = np.linspace(-2.5, 3.6, 500)
mean_pdf = theta_ou + (x0_ou - theta_ou) * np.exp(-lambda_ou * t_pdf)
var_pdf = sigma_ou ** 2 / (2.0 * lambda_ou) * (1.0 - np.exp(-2.0 * lambda_ou * t_pdf))
std_pdf = np.sqrt(var_pdf)
PXT = normal_pdf(x_pdf[None, :], mean_pdf[:, None], std_pdf[:, None])
normalization = integrate(PXT, x_pdf, axis=1)

print(f'PDF normalization range on the plotted grid: '
      f'[{normalization.min():.6f}, {normalization.max():.6f}]')
assert np.allclose(normalization, 1.0, atol=2e-3)

fig_pdf_surface = go.Figure(go.Surface(
    x=x_pdf, y=t_pdf, z=PXT, colorscale='Viridis',
    colorbar=dict(title='p(x,t)')
))
fig_pdf_surface.update_layout(
    title='Evolution of the Ornstein–Uhlenbeck probability density',
    scene=dict(
        xaxis_title='State x',
        yaxis_title='Time t',
        zaxis_title='p(x,t)',
        camera=dict(eye=dict(x=1.45, y=-1.55, z=0.95))
    ),
    template='plotly_white', width=980, height=700
)
fig_pdf_surface.show()


PDF normalization range on the plotted grid: [0.999942, 1.000000]


## Monte Carlo validation of the evolving PDF

The analytical PDF describes the complete ensemble. Histograms from finitely many Euler–Maruyama paths should approach it. Small remaining differences come from finite sampling and time-discretization error.


In [9]:
fig_pdf_validation = make_subplots(
    rows=2, cols=2,
    subplot_titles=[f't = {t:g}' for t in snapshot_times]
)

for index, t_snapshot in enumerate(snapshot_times):
    row = index // 2 + 1
    col = index % 2 + 1
    samples_t = snapshots_ou[t_snapshot]
    mean_t = theta_ou + (x0_ou - theta_ou) * np.exp(-lambda_ou * t_snapshot)
    var_t = sigma_ou ** 2 / (2.0 * lambda_ou) * (1.0 - np.exp(-2.0 * lambda_ou * t_snapshot))
    grid_t = np.linspace(samples_t.min() - 0.3, samples_t.max() + 0.3, 500)

    fig_pdf_validation.add_trace(
        go.Histogram(
            x=samples_t, histnorm='probability density', nbinsx=90,
            opacity=0.55, showlegend=(index == 0), name='Monte Carlo histogram'
        ), row=row, col=col
    )
    fig_pdf_validation.add_trace(
        go.Scatter(
            x=grid_t, y=normal_pdf(grid_t, mean_t, np.sqrt(var_t)),
            mode='lines', line=dict(color='black', width=3),
            showlegend=(index == 0), name='Analytical PDF'
        ), row=row, col=col
    )

fig_pdf_validation.update_xaxes(title_text='State x')
fig_pdf_validation.update_yaxes(title_text='Density')
fig_pdf_validation.update_layout(
    title='Euler–Maruyama ensembles versus the analytical PDF',
    template='plotly_white', width=1_000, height=760, barmode='overlay'
)
fig_pdf_validation.show()


# Final connections

The five sections form one chain:

1. A stochastic process is an indexed family of random variables or vectors.
2. White noise supplies temporally uncorrelated innovations.
3. Its flat PSD is transformed by system dynamics into a colored state spectrum.
4. In discrete time, the dynamics propagate the state mean and covariance.
5. In continuous time, the Langevin SDE propagates random paths while the Fokker–Planck equation propagates their PDF.

For linear systems with Gaussian initial conditions and Gaussian noise, the distribution remains Gaussian. Therefore, tracking only the mean and covariance is exact. This closure property is central to the classical Kalman filter.

## Practical cautions

- Sample autocorrelations and periodograms never look perfectly ideal for finite data.
- A flat PSD does not specify the amplitude distribution.
- Continuous white noise must be interpreted through stochastic integrals.
- Euler–Maruyama introduces discretization error; decreasing $\Delta t$ improves the approximation but increases cost.
- The covariance recursion assumes process noise is independent of the current state unless cross-covariance terms are added.
- Nonlinear or non-Gaussian systems generally require more than the first two moments.

## Exercises

1. Change the eigenvalues of `F_vec`. What happens when one leaves the unit circle?
2. Generate uniform white noise with the same variance. Compare its ACF and PSD with Gaussian white noise.
3. Change `a_ar` from 0.92 to -0.92. Explain why the spectral peak moves toward the Nyquist frequency.
4. Increase the measurement variance `R_measurement`. Which plotted quantities change before a Kalman update is implemented?
5. Halve `dt_ou` and compare the Euler–Maruyama final mean and variance errors.
6. Change `lambda_ou`, `theta_ou`, and `sigma_ou`. Predict the stationary mean and variance before running the code.
7. Replace the OU drift with a nonlinear double-well drift $a(x)=x-x^3$ and estimate the PDF only from Monte Carlo histograms.
